# LIFE on Google Colab — PolitiFact++ (4-class, GPT-2 features)

Runs the LIFE pipeline end-to-end: **convert → key-sentence extraction → concatenate → features → train**.

**Before you start:**
1. Set the Colab runtime to **GPU** (Runtime → Change runtime type → T4 GPU).
2. Upload the whole `LIFE` repo (including `dataset/data/`) to your Google Drive, e.g. `MyDrive/LIFE`.
3. Edit `PROJECT_DIR` in the path cell below if you put it somewhere else.

Scope: **PolitiFact++** only (~520 articles). VLPFN is excluded (its text has no punctuation, so sentence splitting — which the whole method relies on — cannot work). GossipCop++ (~20k) is far heavier; try it only after this works.

In [11]:
# Confirm a GPU is attached
!nvidia-smi

Sat May 30 00:32:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [12]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import os

# <-- change this if you uploaded the repo elsewhere
PROJECT_DIR = '/content/drive/MyDrive/LIFE'
os.chdir(PROJECT_DIR)

POLITIFACT_DIR = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset/PolitiFact++'
OUTPUT_RAW = f'{PROJECT_DIR}/dataset/output_raw'
KEY_SENT   = f'{PROJECT_DIR}/dataset/keySentence/important_sentences_top20.jsonl'
FEATURES   = f'{PROJECT_DIR}/dataset/features'
TRAIN_PATH = f'{PROJECT_DIR}/dataset/train.jsonl'
TEST_PATH  = f'{PROJECT_DIR}/dataset/test.jsonl'

print('cwd:', os.getcwd())
print('PolitiFact++ found:', os.path.isdir(POLITIFACT_DIR))

cwd: /content/drive/MyDrive/LIFE
PolitiFact++ found: True


In [14]:
# Install dependencies.
# If the fastNLP import fails at the training step, pin a compatible version:
#   !pip install -q fastNLP==1.0.1
!pip install -q -r requirements.txt

In [15]:
# NLTK sentence tokenizer data. 'punkt' gives english.pickle (used by train.py);
# 'punkt_tab' is required by newer nltk's sent_tokenize (used by step 1).
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Step 0 — Convert PolitiFact++ JSON to JSONL
Produces `HF_fake.jsonl`, `MF_fake.jsonl`, `HR_true.jsonl`, `MR_true.jsonl` (97 / 97 / 194 / 132 lines).

In [16]:
!python dataset/0_convert.py --input_dir "{POLITIFACT_DIR}" --output_dir "{OUTPUT_RAW}"

HF.json -> HF_fake.jsonl: 97 records (label=human_fake)
MF.json -> MF_fake.jsonl: 97 records (label=gpt3.5_fake)
HR.json -> HR_true.jsonl: 194 records (label=human_true)
MR.json -> MR_true.jsonl: 132 records (label=gpt3.5_true)


## Step 1 — Key-sentence extraction
Trains a BERT fake/real classifier (saved as `gpt3.5_bert_model.pt`), then finds the top-20 most impactful sentences per article. **This is the slowest step** (a forward pass per sentence per article); a few minutes to ~30 min on a T4 for PolitiFact++.

In [17]:
!python dataset/1_keySentenceExtraction.py --data_dir "{OUTPUT_RAW}" --output_file "{KEY_SENT}" --gpu 0

Loading weights: 100% 199/199 [00:00<00:00, 949.24it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing

## Step 2 — Concatenate key sentences back into the data
Adds a `sentence` field to each record in `OUTPUT_RAW` by matching on `(id, label)`. **Overwrites the files in `OUTPUT_RAW` in place** — re-run Step 0 first if you need to reset them.

In [18]:
!python dataset/2_concate.py --folder_path "{OUTPUT_RAW}" --important_sentences_file "{KEY_SENT}"

所有 .jsonl 文件已成功更新。


## Step 3 — Generate GPT-2 fingerprint features (in-process)
Replaces the mosec server + HTTP client with direct GPT-2 log-likelihood scoring. Writes one feature JSONL per input file into `FEATURES`.

In [21]:
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_RAW}" --output_dir "{FEATURES}" --model gpt2 --gpu 0

Streaming output truncated to the last 5000 lines.
34804 34824
34827 34843
34845 34867
1567 1571
34877 34883
34886 34892
34895 34931
34933 34951
34953 34961
641 641
 45% 87/194 [00:09<00:21,  5.00it/s]0 117
118 226
229 285
287 337
340 380
382 446
448 464
467 523
525 535
537 565
637 661
664 682
721 799
802 906
908 922
2247 2295
2298 2338
2341 2373
2376 2420
2422 2436
2439 2499
0 117
305 329
404 452
457 491
582 608
611 625
627 647
650 684
832 884
3335 3353
3358 3396
3398 3430
3435 3509
3514 3620
3625 3649
3651 3705
3707 3749
3754 3800
3802 3830
3835 3903
3908 3920
 46% 89/194 [00:09<00:17,  6.10it/s]0 117
118 222
225 305
308 356
358 457
459 527
530 582
584 626
628 684
687 755
757 787
789 867
870 902
904 1026
1029 1077
0 117
118 195
197 265
268 318
321 343
345 381
383 423
426 472
474 516
519 565
567 603
605 625
627 667
669 727
730 750
752 756
758 808
810 820
0 117
118 144
146 204
255 287
289 313
315 329
331 383
385 407
409 473
678 698
700 716
903 945
5464 5502
5504 5552
5554 5570
5573 560

## Step 4 — Train the classifier
Splits `FEATURES` into train/test, then trains the Transformer+CRF sequence classifier over the 4 classes. Starts with **2 epochs as a smoke test** — raise `--num_train_epochs` (the repo default is 50) once it runs cleanly.

In [24]:
!python LIFE_train/train.py \
  --split_dataset \
  --data_path "{FEATURES}" \
  --train_path "{TRAIN_PATH}" \
  --test_path "{TEST_PATH}" \
  --model Transformer \
  --num_train_epochs 50

Log INFO: split dataset...
********************************
The overall data sources:
['HF_fake.jsonl', 'HR_true.jsonl', 'MF_fake.jsonl', 'MR_true.jsonl']
100% 416/416 [00:00<00:00, 1743.81it/s]
100% 104/104 [00:00<00:00, 1796.82it/s]

The number of train dataset: 416
The number of test  dataset: 104
********************************
100% 416/416 [00:00<00:00, 857.71it/s]
100% 104/104 [00:00<00:00, 6316.91it/s]
--------------------------------classify--------------------------------
Log INFO: do train...
Epoch:   0% 0/50 [00:00<?, ?it/s]
Iteration:   0% 0/13 [00:00<?, ?it/s]
Iteration:   8% 1/13 [00:00<00:07,  1.64it/s]
Iteration:  15% 2/13 [00:00<00:03,  2.88it/s]
Iteration:  23% 3/13 [00:00<00:02,  3.94it/s]
Iteration:  31% 4/13 [00:01<00:01,  4.68it/s]
Iteration:  38% 5/13 [00:01<00:01,  5.31it/s]
Iteration:  46% 6/13 [00:01<00:01,  5.58it/s]
Iteration:  54% 7/13 [00:01<00:01,  5.75it/s]
Iteration:  62% 8/13 [00:01<00:00,  5.78it/s]
Iteration:  69% 9/13 [00:01<00:00,  6.22it/s]
Itera

## Notes / troubleshooting
- **fastNLP**: if step 4 errors on `from fastNLP.modules.torch import ...`, run `!pip install -q fastNLP==1.0.1` and restart the runtime.
- **Checkpoints**: `gpt3.5_bert_model.pt` (step 1) and `linear_en.pt` (step 4) are written to `PROJECT_DIR` on Drive, so they survive disconnects.
- **Scaling up**: point Step 0 at `.../Dataset/GossipCop++` and rerun — but expect Step 1 to take hours on a T4.
- **Re-runs**: Step 2 mutates `OUTPUT_RAW` in place; always re-run Step 0 before re-running Steps 1–3 from scratch.